# Benchmark Model Training (Self-Contained)

This notebook trains and evaluates baseline models for emotion classification.

**Objective**: Establish performance baseline with simple models - a minimum viable product.

**Data Source**: Loads `Combined_Emotion_Data.csv` directly (no need to run notebooks 05 or 09)

**Models**:
1. **Logistic Regression** - Simple linear baseline
2. **XGBoost** - Non-linear tree-based baseline

**Workflow**:
1. Load data from local CSV or S3
2. Basic preprocessing (scaling, encoding)
3. Train/test split (80/20)
4. Train both models
5. Evaluate and compare
6. Save models to S3

**Evaluation**: Focus on **Macro F1-score** due to class imbalance.

## Setup and Imports

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from time import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

print(f"SageMaker version: {sagemaker.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Initialize SageMaker session and S3 client
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
s3_client = boto3.client('s3', region_name=region)

print(f"Region: {region}")
print(f"Bucket: {bucket}")
print(f"Role: {role}")

In [ ]:
# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# S3 paths for saving models
s3_benchmarks_prefix = f"s3://{bucket}/models/benchmarks"

print(f"Benchmarks will be saved to: {s3_benchmarks_prefix}")

## Load and Prepare Data

In [ ]:
# Load Combined_Emotion_Data.csv
local_file = "Datasets/Combined_Emotion_Data.csv"

if not os.path.exists(local_file):
    print(f"[ERROR] File not found: {local_file}")
    print("\nPlease ensure Combined_Emotion_Data.csv exists in the Datasets/ directory")
    print("You can obtain this file from a teammate or run notebook 05 (takes 3+ hours)")
    raise FileNotFoundError(f"Missing required file: {local_file}")

print(f"Loading data from {local_file}...")
df = pd.read_csv(local_file)

print(f"\n[OK] Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Check data quality
print("Data Quality Check:\n")

# Missing values
missing = df.isnull().sum()
print(f"Missing values: {missing.sum()} total")
if missing.sum() > 0:
    print(missing[missing > 0])

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Label distribution
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

## Data Preprocessing

In [ ]:
# Standardize labels to title case (fixes 'sad' vs 'Sad' issue)
df['label'] = df['label'].str.title()

print("Labels after standardization:")
print(df['label'].value_counts().sort_index())
print(f"\nTotal samples: {len(df)}")

In [ ]:
# Define feature columns (20 acoustic features from the dataset)
feature_cols = [
    'meanfreq', 'sd', 'median', 'q25', 'q75', 'iqr', 'skew', 'kurt',
    'sp_ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun',
    'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx'
]

print(f"Using {len(feature_cols)} acoustic features:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

## Label Encoding and Feature Scaling

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

print("Label encoding:")
for label, code in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    count = (df['label'] == label).sum()
    print(f"  {code}: {label:12s} ({count:5d} samples, {count/len(df)*100:5.2f}%)")

In [ ]:
## Train/Test Split (80/20)

# Stratified split to maintain class distribution
X = df[feature_cols]
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"Train/Test Split:")
print(f"  Training: {len(X_train):5d} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Test:     {len(X_test):5d} samples ({len(X_test)/len(df)*100:.1f}%)")
print(f"  Total:    {len(df):5d} samples")

In [ ]:
# Apply StandardScaler to normalize features
scaler = StandardScaler()

# Fit on training data only, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied (StandardScaler)")
print(f"  Mean ≈ 0, Std ≈ 1 for all features")
print(f"\nScaled training data shape: {X_train_scaled.shape}")

In [ ]:
# Verify stratification - class distribution should be same in train/test
print("Class distribution verification:\n")

train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_dist = pd.Series(y_test).value_counts(normalize=True).sort_index()

print("Training set:")
for class_id in range(len(label_encoder.classes_)):
    emotion = label_encoder.classes_[class_id]
    train_pct = train_dist.get(class_id, 0) * 100
    test_pct = test_dist.get(class_id, 0) * 100
    print(f"  {class_id}: {emotion:12s} - Train: {train_pct:5.2f}%, Test: {test_pct:5.2f}%")

# Calculate class imbalance ratio
class_counts = pd.Series(y_train).value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}x")
print("[NOTE] Using balanced class weights and Macro F1-score to handle imbalance")

## Define Evaluation Functions

In [ ]:
def evaluate_model(y_true, y_pred, model_name, label_encoder):
    """
    Comprehensive evaluation for multiclass classification.
    Returns metrics dict, classification report, and confusion matrix.
    """
    # Calculate metrics
    metrics = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_per_class': f1_score(y_true, y_pred, average=None).tolist()
    }
    
    # Classification report
    report = classification_report(
        y_true, y_pred,
        target_names=label_encoder.classes_,
        output_dict=True
    )
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    return metrics, report, cm

In [ ]:
def plot_confusion_matrix(cm, class_names, title, save_path=None):
    """
    Plot confusion matrix as heatmap (normalized by row).
    """
    # Normalize by row (true class)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        cbar_kws={'label': 'Proportion'},
        ax=ax
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    
    plt.show()

In [ ]:
def print_metrics(metrics, report):
    """
    Print formatted evaluation metrics.
    """
    print(f"\n{'='*60}")
    print(f"Model: {metrics['model_name']}")
    print(f"{'='*60}\n")
    
    print("Overall Metrics:")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  F1 (Macro):        {metrics['f1_macro']:.4f} ← PRIMARY METRIC")
    print(f"  F1 (Weighted):     {metrics['f1_weighted']:.4f}")
    print(f"  Precision (Macro): {metrics['precision_macro']:.4f}")
    print(f"  Recall (Macro):    {metrics['recall_macro']:.4f}")
    
    print(f"\nPer-Class F1 Scores:")
    for i, (class_name, f1) in enumerate(zip(label_encoder.classes_, metrics['f1_per_class'])):
        print(f"  {i}: {class_name:12s} - {f1:.4f}")
    
    print(f"\n{'='*60}")

## Model 1: Logistic Regression

In [ ]:
print("Training Logistic Regression (One-vs-Rest with balanced class weights)...\n")

start_time = time()

lr_model = LogisticRegression(
    multi_class='ovr',  # One-vs-Rest for 7 classes
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1  # Use all CPU cores
)

lr_model.fit(X_train_scaled, y_train)

lr_train_time = time() - start_time

print(f"[OK] Logistic Regression trained in {lr_train_time:.2f} seconds")

In [ ]:
# Make predictions on test set
print("Making predictions on test set...\n")

start_time = time()
y_pred_lr = lr_model.predict(X_test_scaled)
lr_inference_time = time() - start_time

print(f"Predictions completed in {lr_inference_time:.4f} seconds")
print(f"Average inference time: {lr_inference_time / len(X_test_scaled) * 1000:.2f} ms per sample")

### Logistic Regression Evaluation

In [ ]:
# Evaluate Logistic Regression
lr_metrics, lr_report, lr_cm = evaluate_model(
    y_test, y_pred_lr, 'Logistic Regression', label_encoder
)

# Add timing metrics
lr_metrics['train_time_sec'] = lr_train_time
lr_metrics['inference_time_sec'] = lr_inference_time

print_metrics(lr_metrics, lr_report)

In [ ]:
# Plot confusion matrix for Logistic Regression
plot_confusion_matrix(
    lr_cm, 
    label_encoder.classes_, 
    'Logistic Regression - Confusion Matrix (Normalized)',
    save_path='/tmp/lr_confusion_matrix.png'
)

In [ ]:
# Display classification report
print("\nDetailed Classification Report:\n")
print(classification_report(
    y_test, y_pred_lr,
    target_names=label_encoder.classes_,
    digits=4
))

In [ ]:
# Analyze feature coefficients (top features per class)
print("Top 5 features per class (by absolute coefficient):\n")

coefficients_df = pd.DataFrame(
    lr_model.coef_,
    index=label_encoder.classes_,
    columns=feature_cols
)

for emotion in label_encoder.classes_:
    top_features = coefficients_df.loc[emotion].abs().sort_values(ascending=False).head(5)
    print(f"{emotion}:")
    for feature, coef_abs in top_features.items():
        coef = coefficients_df.loc[emotion, feature]
        print(f"  {feature:20s}: {coef:+.4f}")
    print()

# Save coefficients
coefficients_df.T.to_csv('/tmp/lr_coefficients.csv')
print("Coefficients saved to /tmp/lr_coefficients.csv")

## Model 2: XGBoost

In [ ]:
print("Training XGBoost Classifier...\n")

start_time = time()

# Calculate sample weights for class imbalance
sample_weights = compute_sample_weight('balanced', y_train)

xgb_model = XGBClassifier(
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=RANDOM_SEED,
    use_label_encoder=False,
    n_jobs=-1
)

# Train with sample weights for class balance
xgb_model.fit(
    X_train_scaled, y_train,
    sample_weight=sample_weights,
    verbose=False
)

xgb_train_time = time() - start_time

print(f"[OK] XGBoost trained in {xgb_train_time:.2f} seconds")

In [ ]:
# Make predictions on test set
print("Making predictions on test set...\n")

start_time = time()
y_pred_xgb = xgb_model.predict(X_test_scaled)
xgb_inference_time = time() - start_time

print(f"Predictions completed in {xgb_inference_time:.4f} seconds")
print(f"Average inference time: {xgb_inference_time / len(X_test_scaled) * 1000:.2f} ms per sample")

### XGBoost Evaluation

In [ ]:
# Evaluate XGBoost
xgb_metrics, xgb_report, xgb_cm = evaluate_model(
    y_test, y_pred_xgb, 'XGBoost', label_encoder
)

# Add timing metrics
xgb_metrics['train_time_sec'] = xgb_train_time
xgb_metrics['inference_time_sec'] = xgb_inference_time

print_metrics(xgb_metrics, xgb_report)

In [ ]:
# Plot confusion matrix for XGBoost
plot_confusion_matrix(
    xgb_cm, 
    label_encoder.classes_, 
    'XGBoost - Confusion Matrix (Normalized)',
    save_path='/tmp/xgb_confusion_matrix.png'
)

In [ ]:
# Display classification report
print("\nDetailed Classification Report:\n")
print(classification_report(
    y_test, y_pred_xgb,
    target_names=label_encoder.classes_,
    digits=4
))

In [ ]:
# Feature importance analysis
print("Feature Importance (Top 15 features):\n")

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15).to_string(index=False))

# Save feature importance
feature_importance.to_csv('/tmp/xgb_feature_importance.csv', index=False)
print("\nFeature importance saved to /tmp/xgb_feature_importance.csv")

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))

top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('XGBoost - Top 15 Feature Importances', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'Precision (Macro)', 'Recall (Macro)', 'Training Time (s)', 'Inference Time (s)'],
    'Logistic Regression': [
        lr_metrics['accuracy'],
        lr_metrics['f1_macro'],
        lr_metrics['f1_weighted'],
        lr_metrics['precision_macro'],
        lr_metrics['recall_macro'],
        lr_metrics['train_time_sec'],
        lr_metrics['inference_time_sec']
    ],
    'XGBoost': [
        xgb_metrics['accuracy'],
        xgb_metrics['f1_macro'],
        xgb_metrics['f1_weighted'],
        xgb_metrics['precision_macro'],
        xgb_metrics['recall_macro'],
        xgb_metrics['train_time_sec'],
        xgb_metrics['inference_time_sec']
    ]
})

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80 + "\n")
print(comparison_df.to_string(index=False))
print("\n" + "="*80)

In [ ]:
# Per-class F1 comparison
per_class_comparison = pd.DataFrame({
    'Emotion': label_encoder.classes_,
    'Logistic Regression': lr_metrics['f1_per_class'],
    'XGBoost': xgb_metrics['f1_per_class']
})

print("\nPer-Class F1-Score Comparison:\n")
print(per_class_comparison.to_string(index=False))

# Calculate improvements
per_class_comparison['Improvement'] = per_class_comparison['XGBoost'] - per_class_comparison['Logistic Regression']
print(f"\nAverage improvement: {per_class_comparison['Improvement'].mean():.4f}")

In [ ]:
# Visualize per-class F1 comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(label_encoder.classes_))
width = 0.35

bars1 = ax.bar(x - width/2, lr_metrics['f1_per_class'], width, label='Logistic Regression', alpha=0.7)
bars2 = ax.bar(x + width/2, xgb_metrics['f1_per_class'], width, label='XGBoost', alpha=0.7)

ax.set_xlabel('Emotion Class', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Per-Class F1-Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(label_encoder.classes_, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig('/tmp/per_class_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Models and Artifacts

In [ ]:
print("Saving models and artifacts to S3...\n")

# Save Logistic Regression model and artifacts
print("Saving Logistic Regression artifacts...")

with open('/tmp/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
s3_client.upload_file('/tmp/lr_model.pkl', bucket, 'models/benchmarks/logistic_regression/model.pkl')
print("  Model: s3://{}/models/benchmarks/logistic_regression/model.pkl".format(bucket))

with open('/tmp/lr_metrics.json', 'w') as f:
    json.dump(lr_metrics, f, indent=2)
s3_client.upload_file('/tmp/lr_metrics.json', bucket, 'models/benchmarks/logistic_regression/metrics.json')
print("  Metrics: s3://{}/models/benchmarks/logistic_regression/metrics.json".format(bucket))

s3_client.upload_file('/tmp/lr_confusion_matrix.png', bucket, 'models/benchmarks/logistic_regression/confusion_matrix.png')
print("  Confusion matrix: s3://{}/models/benchmarks/logistic_regression/confusion_matrix.png".format(bucket))

s3_client.upload_file('/tmp/lr_coefficients.csv', bucket, 'models/benchmarks/logistic_regression/coefficients.csv')
print("  Coefficients: s3://{}/models/benchmarks/logistic_regression/coefficients.csv".format(bucket))

print("\n[OK] Logistic Regression artifacts saved")

In [ ]:
# Save XGBoost model and artifacts
print("\nSaving XGBoost artifacts...")

with open('/tmp/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
s3_client.upload_file('/tmp/xgb_model.pkl', bucket, 'models/benchmarks/xgboost/model.pkl')
print("  Model: s3://{}/models/benchmarks/xgboost/model.pkl".format(bucket))

with open('/tmp/xgb_metrics.json', 'w') as f:
    json.dump(xgb_metrics, f, indent=2)
s3_client.upload_file('/tmp/xgb_metrics.json', bucket, 'models/benchmarks/xgboost/metrics.json')
print("  Metrics: s3://{}/models/benchmarks/xgboost/metrics.json".format(bucket))

s3_client.upload_file('/tmp/xgb_confusion_matrix.png', bucket, 'models/benchmarks/xgboost/confusion_matrix.png')
print("  Confusion matrix: s3://{}/models/benchmarks/xgboost/confusion_matrix.png".format(bucket))

s3_client.upload_file('/tmp/xgb_feature_importance.csv', bucket, 'models/benchmarks/xgboost/feature_importance.csv')
print("  Feature importance (CSV): s3://{}/models/benchmarks/xgboost/feature_importance.csv".format(bucket))

s3_client.upload_file('/tmp/xgb_feature_importance.png', bucket, 'models/benchmarks/xgboost/feature_importance.png')
print("  Feature importance (PNG): s3://{}/models/benchmarks/xgboost/feature_importance.png".format(bucket))

print("\n[OK] XGBoost artifacts saved")

In [ ]:
# Save benchmark comparison
print("\nSaving benchmark comparison...")

# Determine best model based on macro F1
best_model = 'xgboost' if xgb_metrics['f1_macro'] > lr_metrics['f1_macro'] else 'logistic_regression'

comparison = {
    'timestamp': datetime.now().isoformat(),
    'random_seed': RANDOM_SEED,
    'dataset': {
        'total_samples': len(df),
        'train_samples': len(X_train_scaled),
        'test_samples': len(X_test_scaled),
        'num_features': len(feature_cols),
        'num_classes': len(label_encoder.classes_)
    },
    'models': {
        'logistic_regression': lr_metrics,
        'xgboost': xgb_metrics
    },
    'best_model': best_model,
    'best_f1_macro': max(lr_metrics['f1_macro'], xgb_metrics['f1_macro']),
    'recommendation': f"Use {best_model.replace('_', ' ').title()} as baseline for further tuning"
}

with open('/tmp/benchmark_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

s3_client.upload_file('/tmp/benchmark_comparison.json', bucket, 'models/benchmarks/benchmark_comparison.json')
print("  Comparison: s3://{}/models/benchmarks/benchmark_comparison.json".format(bucket))

print("\n[OK] All artifacts saved to S3")

## Verification

In [ ]:
# Verify S3 uploads
print("Verifying S3 uploads...\n")

response = s3_client.list_objects_v2(
    Bucket=bucket,
    Prefix='models/benchmarks/'
)

print("Files uploaded to S3:\n")
for obj in response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    print(f"  {obj['Key']:60s} ({size_kb:8.2f} KB)")

print(f"\nTotal files: {len(response.get('Contents', []))}")

In [ ]:
# Test loading saved models
print("\nTesting model loading...\n")

# Test XGBoost model
s3_client.download_file(bucket, 'models/benchmarks/xgboost/model.pkl', '/tmp/test_xgb.pkl')
with open('/tmp/test_xgb.pkl', 'rb') as f:
    loaded_xgb = pickle.load(f)

test_predictions = loaded_xgb.predict(X_test_scaled[:10])
original_predictions = xgb_model.predict(X_test_scaled[:10])

if np.array_equal(test_predictions, original_predictions):
    print("[OK] XGBoost model loaded successfully and predictions match")
else:
    print("[ERROR] XGBoost model predictions do not match!")

# Test Logistic Regression model
s3_client.download_file(bucket, 'models/benchmarks/logistic_regression/model.pkl', '/tmp/test_lr.pkl')
with open('/tmp/test_lr.pkl', 'rb') as f:
    loaded_lr = pickle.load(f)

test_predictions = loaded_lr.predict(X_test_scaled[:10])
original_predictions = lr_model.predict(X_test_scaled[:10])

if np.array_equal(test_predictions, original_predictions):
    print("[OK] Logistic Regression model loaded successfully and predictions match")
else:
    print("[ERROR] Logistic Regression model predictions do not match!")

## Summary and Next Steps

In [ ]:
print("="*80)
print("BENCHMARK MODEL TRAINING - COMPLETE")
print("="*80)

print("\nDataset Summary:")
print(f"  Total samples:      {len(df):,}")
print(f"  Training samples:   {len(X_train_scaled):,} (80%)")
print(f"  Test samples:       {len(X_test_scaled):,} (20%)")
print(f"  Features:           {len(feature_cols)}")
print(f"  Classes:            {len(label_encoder.classes_)}")

print("\nModel Performance (Test Set):")
print(f"\n  Logistic Regression:")
print(f"    Accuracy:    {lr_metrics['accuracy']:.4f}")
print(f"    F1 (Macro):  {lr_metrics['f1_macro']:.4f}")
print(f"    Train time:  {lr_metrics['train_time_sec']:.2f}s")

print(f"\n  XGBoost:")
print(f"    Accuracy:    {xgb_metrics['accuracy']:.4f}")
print(f"    F1 (Macro):  {xgb_metrics['f1_macro']:.4f}")
print(f"    Train time:  {xgb_metrics['train_time_sec']:.2f}s")

if xgb_metrics['f1_macro'] > lr_metrics['f1_macro']:
    improvement = (xgb_metrics['f1_macro'] - lr_metrics['f1_macro']) / lr_metrics['f1_macro'] * 100
    print(f"\n  XGBoost improvement over Logistic Regression: {improvement:+.2f}%")

print("\nBest Model (by Macro F1):")
print(f"  {comparison['best_model'].replace('_', ' ').title()}")
print(f"  F1 (Macro): {comparison['best_f1_macro']:.4f}")

print("\nKey Findings:")
print("  ✓ Both models trained successfully on local data")
print(f"  ✓ {'XGBoost' if xgb_metrics['f1_macro'] > lr_metrics['f1_macro'] else 'Logistic Regression'} performs better on Macro F1")

if max(xgb_metrics['f1_macro'], lr_metrics['f1_macro']) > 0.50:
    print(f"  ✓ Best model achieves >0.50 macro F1 (better than random)")
else:
    print(f"  ⚠ Macro F1 below 0.50 - consider feature engineering")

print("  ✓ All artifacts saved to S3")
print("  ✓ Models verified by successful loading and prediction")

print("\nClass-Specific Observations:")
worst_class_idx = np.argmin(xgb_metrics['f1_per_class'])
worst_class = label_encoder.classes_[worst_class_idx]
worst_f1 = xgb_metrics['f1_per_class'][worst_class_idx]
print(f"  Weakest class: {worst_class} (F1: {worst_f1:.4f})")

print("\nS3 Locations:")
print(f"  Benchmarks: s3://{bucket}/models/benchmarks/")
print(f"    - logistic_regression/")
print(f"    - xgboost/")
print(f"    - benchmark_comparison.json")

print("\nNext Steps:")
print("  1. Review model performance and confusion matrices")
print("  2. Hyperparameter tuning to improve performance")
print("  3. Deploy best model to SageMaker endpoint")
print("  4. Set up monitoring and CI/CD pipeline")

print("\n" + "="*80)

## Release Resources

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}